In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/20/26 14:23:22] INFO     Using                                                                  ]8;id=572350;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=9389;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[04/20/26 14:23:23] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\requests\__ini ]8;id=227501;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=846142;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             t__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                     
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[04/20/26 14:23:24] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=350397;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=730330;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Fuentes de información

In [3]:
central_calendario_extendido_uptoday = catalog.load('central_calendario_extendido_uptoday')
central_bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')
central_graduados_calendario_academico = catalog.load('central_graduados_calendario_academico')
central_activos_calendario = catalog.load('central_activos_calendario')
central_estados_calac = catalog.load('central_estados_calac')


                    INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=250079;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=459521;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

[04/20/26 14:23:25] INFO     Loading data from central_bajas_calendario_academico              ]8;id=90285;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=306421;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_graduados_calendario_academico          ]8;id=703783;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=29954;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_activos_calendario (ParquetDataset)...  ]8;id=968764;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=691955;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from central_estados_calac (ParquetDataset)...       ]8;id=143430;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=905340;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

# Parámetros

In [4]:
parameters = catalog.load('parameters')
dict_niveles_duracion = parameters['graduados_calac']['dict_niveles_duracion']

                    INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=205663;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=283504;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [5]:
# parámetros 
keys = [
    'cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel', 
    'programa', 'fecha_inicio', 'fecha_fin', 'semana', 
    'semana_acumulada', 'month', 'mes_academico', 'student_journey'
]

columnas_to_order = ['fecha_ingreso','month','fecha_inicio', 'fecha_fin']

group_keys = [
    'cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel', 'programa'
]


survival_columns = ['cohorte_inicial',
                    'fecha_ingreso',
                    'nivel_academico',
                    'nivel',
                    'programa',
                    'fecha_inicio',
                    'fecha_fin', 
                    'semana',
                    'semana_acumulada',
                    
                    'semana_limite',
                    'month',
                    'mes_academico',
                    'student_journey',
                    'nuevos',
                    'di',
                    'gi',
                    'engi', 
                    'ci',
                    'di_cum',
                    'gi_cum', 
                    'ai', 
                    'ni']

# Lifetables
- Cascadas 
- Sobrevivencia

## Semanal

### Estudiantes Nuevos

In [6]:
# Punto de inicio semana y mes igual a 0 
central_count_nuevos = (central_estados_calac.loc[:, ['cohorte_inicial',
                                                      'fecha_ingreso',
                                                      'nivel_academico', 
                                                      'nivel',
                                                      'programa',
                                                      'identificacion']]
                            .groupby(['cohorte_inicial',
                                      'fecha_ingreso',
                                      'nivel_academico', 
                                      'nivel',
                                      'programa'])
                            .agg(nuevos= ('identificacion','count'))
                            .reset_index()
                            .sort_values(by=['fecha_ingreso','nuevos'], ascending=[True,False])
                            )

def generar_cascada_con_punto_cero(central_count_nuevos, central_calendario_extendido_uptoday):
    # 1. Crear el punto de partida (T=0)
    punto_cero = central_count_nuevos.copy()
    
    punto_cero['fecha_inicio'] = punto_cero['fecha_ingreso']
    punto_cero['fecha_fin'] = punto_cero['fecha_ingreso']
    punto_cero['semana'] = 0
    punto_cero['semana_acumulada'] = 0
    punto_cero['month'] = 0
    punto_cero['mes_academico'] = 'm0'
    punto_cero['student_journey'] = 'ingreso'

    # 2. Tu merge original (El recorrido futuro de 1 a N)
    recorrido_futuro = pd.merge(
        central_count_nuevos, 
        central_calendario_extendido_uptoday[[
            'cohorte_inicial', 'fecha_ingreso', 'fecha_inicio', 
            'fecha_fin', 'semana', 'semana_acumulada', 
            'month', 'mes_academico', 'student_journey'
        ]],
        on=['cohorte_inicial', 'fecha_ingreso'],  
        how='left'
    )

    # 3. Concatenar el Punto Cero con el Recorrido
    # Usamos ignore_index=True para mantener el índice limpio
    cascadas_completa = pd.concat([punto_cero, recorrido_futuro], ignore_index=True)

    # 4. Ordenar para que el 'ingreso' (0) quede al principio de cada grupo
    cascadas_completa = cascadas_completa.sort_values(
        by=['cohorte_inicial', 'programa', 'semana_acumulada']
    )

    return cascadas_completa

cascadas_inicial_calendario  = generar_cascada_con_punto_cero(central_count_nuevos, central_calendario_extendido_uptoday)

### Bajas

In [7]:
# Conteo Bajas por semana 
central_count_bajas = (central_bajas_calendario_academico.loc[:, keys + ['di']]
                       .groupby(keys)
                       .agg({'di': 'sum'})
                       .reset_index()
                       .sort_values(by=columnas_to_order, ascending=True))


In [8]:
# Realizar el merge
cascadas_semanal = pd.merge(
    cascadas_inicial_calendario,
    central_count_bajas[keys + ['di']], # Solo traemos las llaves y la métrica
    on=keys,
    how='left'
)

# Llenar vacíos: si no hubo baja esa semana, es 0
cascadas_semanal['di'] = cascadas_semanal['di'].fillna(0)

### Graduados

In [9]:
central_count_graduados = (central_graduados_calendario_academico.loc[:, keys + ['gi']]
                            .groupby(keys)
                            .agg({'gi': 'sum'})
                            .reset_index()
                            .sort_values(by=['fecha_ingreso','month','fecha_inicio', 'fecha_fin'], ascending=True))


In [10]:
# Supongamos que tienes un central_count_graduados con la columna 'gi'
cascadas_semanal = pd.merge(
    cascadas_semanal,
    central_count_graduados[keys + ['gi']], 
    on=keys,
    how='left'
)
cascadas_semanal['gi'] = cascadas_semanal['gi'].fillna(0)

# Construcción de las tablas de vida 

## Podas

In [11]:
def podar_tabla_vida(df: pd.DataFrame, dict_duracion: dict) -> pd.DataFrame:
    """
    Filtra las semanas que exceden la duración teórica del programa.
    """
    
    def obtener_limite(row):
        # Normalización robusta
        nivel = str(row['nivel']).lower().strip()
        programa = str(row['programa']).lower().strip()
        
        # Acceso seguro al diccionario
        nivel_dict = dict_duracion.get(nivel, {})
        
        # Prioridad 1: Programa específico
        if programa in nivel_dict:
            return nivel_dict[programa].get('semanas', float('inf'))
        
        # Prioridad 2: Default del nivel
        if 'default' in nivel_dict:
            return nivel_dict['default'].get('semanas', float('inf'))
        
        # Prioridad 3: Estructura plana (especialización)
        return nivel_dict.get('semanas', float('inf'))

    # Aplicamos el cálculo del límite
    df['limite_semanas'] = df.apply(obtener_limite, axis=1)
    
    # Filtrado: mantenemos solo lo que está dentro del rango
    df_podado = df[df['semana_acumulada'] <= df['limite_semanas']].copy()
    
    # Limpieza
    return df_podado.drop(columns=['limite_semanas'])

# Calcular las deserciones acumuladas, las graduaciones acumuladas y los ai: estudiantes activos

In [12]:
cascadas_semanal_podada = podar_tabla_vida(cascadas_semanal,dict_niveles_duracion)
# 5. Calcular la baja acumulada (di_cum) por cohorte, nivel y programa ordenado por fecha_inicio, fecha_fin, semana_acumulada y month
cascadas_semanal_podada['di_cum'] = cascadas_semanal_podada.groupby(['fecha_ingreso',
                                                        'nivel_academico',
                                                        'nivel',
                                                        'programa'
                                    ])['di'].cumsum()

cascadas_semanal_podada['gi_cum'] = cascadas_semanal_podada.groupby(['fecha_ingreso',
                                                        'nivel_academico',
                                                        'nivel',
                                                        'programa'])['gi'].cumsum()

# 6. Calcular la cantidad de estudiantes activos restando los ingresos menos la baja acumulada
cascadas_semanal_podada['ai'] = cascadas_semanal_podada['nuevos'] - cascadas_semanal_podada['di_cum'] - cascadas_semanal_podada['gi_cum']

In [13]:
# 1. Definir las columnas de agrupación (tus llaves de negocio)
group_keys = [
    'cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel', 'programa'
]

# 2. Asegurar el orden cronológico antes de hacer el shift
cascadas_semanal_podada = cascadas_semanal_podada.sort_values(
    by=group_keys + ['semana_acumulada']
)

In [14]:
def aplicar_logica_semana_final(df, dict_niveles):
    def obtener_semana_final(row):
        # Normalizamos los valores para evitar fallos por mayúsculas/minúsculas o espacios
        nivel = str(row['nivel']).lower().strip()
        programa = str(row['programa']).lower().strip()
        
        # 1. Caso Especialización (estructura plana)
        if nivel == 'especializacion':
            return dict_niveles['especializacion']['default']['semanas']
        
        # 2. Casos con excepciones (Maestría y Pregrado)
        if nivel in dict_niveles:
            config = dict_niveles[nivel]
            if programa in config:
                return config[programa]['semanas']
            else:
                return config.get('default', {}).get('semanas', np.nan)
        
        return np.nan

    # Calculamos la semana final para cada registro según el diccionario
    # Se crea una columna temporal 'semana_limite'
    df['semana_limite'] = df.apply(obtener_semana_final, axis=1)

    # Condición: si la 'semana_acumulada' (o 'semana') es igual o mayor a la semana_limite
    # Nota: Uso 'semana_acumulada' porque suele ser el indicador del progreso total.
    mask = df['semana_acumulada'] >= df['semana_limite']
    
    # Asignamos ci = estudiantes_activos donde se cumple la condición
    df.loc[mask, 'engi'] = df.loc[mask, 'ai']
    
    # Opcional: Eliminar la columna auxiliar para dejar el DF limpio
    # df.drop(columns=['semana_limite'], inplace=True)
    
    return df

In [15]:
cascadas_semanal_podada = aplicar_logica_semana_final(cascadas_semanal_podada, dict_niveles_duracion)
mask_engi = cascadas_semanal_podada['engi'] >= cascadas_semanal_podada['gi']
cascadas_semanal_podada.loc[mask_engi,'engi'] = cascadas_semanal_podada['engi'] - cascadas_semanal_podada['gi'] 
cascadas_semanal_podada['engi'] = cascadas_semanal_podada['engi'].fillna(0)

In [165]:
def calcular_censuras_academica(df: pd.DataFrame, id_cols: list) -> pd.DataFrame:
    """
    Ajusta ni, di y ci para el análisis de supervivencia basado en el 
    recorrido semanal académico.
    """
    # 1. Asegurar orden cronológico estricto
    df = df.sort_values(by=id_cols + ['semana_acumulada'])

    # 2. Identificar el último punto observado en la data para cada estudiante/programa
    # (Esto captura tanto a los que terminaron por 'poda' como a los que llegan a hoy)
    df['es_ultimo_punto'] = (
        df.groupby(id_cols)['semana_acumulada'].transform('max') == df['semana_acumulada']
    )

    # 3. ni (En Riesgo): Ya lo calculamos como el valor al inicio de la semana.
    # Si por alguna razón no viene en el DF, ni es el 'ai' del periodo anterior.
    # (Asumimos que ya tienes 'ni' del paso anterior, si no, descomenta la siguiente línea)
    df['ni'] = df.groupby(id_cols)['ai'].shift(1).fillna(df['nuevos'])

    # 4. di (Eventos - Bajas):
    # Usamos el 'di' que ya calculaste (conteo de identificaciones con baja).
    # IMPORTANTE: En el último punto observado, no registramos bajas nuevas 
    # porque no podemos confirmar si ocurrió el evento o se acabó el tiempo de observación.
    #df['di_final'] = np.where(~df['es_ultimo_punto'], df['di'], 0)

    # 5. ci (Censuras):
    # Son los estudiantes que permanecen activos (ai) en el último punto 
    # observado del dataset. 
    df['ci'] = np.where(df['es_ultimo_punto'], df['ai'] + df['gi'], 0)

    # 6. Limpieza y preparación final
    # Eliminamos columnas auxiliares para no ensuciar el catálogo
    columnas_finales = df.drop(columns=['es_ultimo_punto'])
    
    return columnas_finales

cascadas_semanal_podada_censuras =  calcular_censuras_academica(cascadas_semanal_podada,group_keys)

In [166]:
mask_programa =  cascadas_semanal_podada_censuras['programa'] == 'auditoria y control'
mask_cohorte = cascadas_semanal_podada_censuras['cohorte_inicial'] == '2025 1a'
cascadas_semanal_podada_censuras[mask_cohorte & mask_programa]

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,nuevos,fecha_inicio,fecha_fin,semana,semana_acumulada,...,student_journey,di,gi,di_cum,gi_cum,ai,semana_limite,engi,ni,ci
108,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-03-17,2025-03-17,0,0,...,ingreso,0.0,0.0,0.0,0.0,29.0,32,0.0,29.0,0.0
109,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-03-17,2025-03-23,1,1,...,onboarding,0.0,0.0,0.0,0.0,29.0,32,0.0,29.0,0.0
110,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-03-24,2025-03-30,2,2,...,onboarding,0.0,0.0,0.0,0.0,29.0,32,0.0,29.0,0.0
111,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-03-31,2025-04-06,3,3,...,onboarding,2.0,0.0,2.0,0.0,27.0,32,0.0,29.0,0.0
112,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-04-07,2025-04-13,4,4,...,onboarding,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0
113,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-04-14,2025-04-20,5,5,...,onboarding,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0
114,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-04-21,2025-04-27,6,6,...,onboarding,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0
115,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-04-28,2025-05-04,7,7,...,onboarding,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0
116,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-05-05,2025-05-11,8,8,...,onboarding,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0
117,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,29,2025-05-12,2025-05-18,1,9,...,q1,0.0,0.0,2.0,0.0,27.0,32,0.0,27.0,0.0


In [200]:
import pandas as pd
import numpy as np

def calcular_km_y_eti_dinamico(
    df: pd.DataFrame, 
    group_cols: list, 
    conf_z: float = 1.96
) -> pd.DataFrame:
    """
    Calcula Kaplan-Meier y ETI con lógica de flujo corregida.
    di: desertores (eventos)
    gi/engi: graduados/egresados (censuras competitivas o salidas administrativas)
    ci: censuras por cierre de cohorte (administrativas)
    """
    
    # 1. Agregación dinámica
    # 1. Agregación dinámica
    cols_metrica = ['di', 'gi', 'engi', 'ci', 'nuevos']
    df_agrupado = (
        df.groupby(group_cols + ['semana_acumulada'])
        .agg({col: 'sum' for col in cols_metrica})
        .reset_index()
    )


    # 2. Ordenar
    df_agrupado = df_agrupado.sort_values(by=group_cols + ['semana_acumulada'])

    # 3. Flujo de población (Cálculo de ni y ai)
    # n_total: Estudiantes que inician en la semana 0 para cada grupo
    df_agrupado['n_total'] = df_agrupado.groupby(group_cols + ['semana_acumulada'] )['nuevos'].transform('sum')

    def calcular_flujos(group):
        # ni (En riesgo): Al inicio de la primera semana es el total.
        # Para semanas posteriores, es lo que quedó de la anterior (ai_prev)
        # menos los que se censuraron administrativamente (ci) al final de la anterior.
        
        # di: desertores (evento)
        # gi + engi + ci: censuras (salidas que no son el evento de interés)
        
        # Calculamos acumulados de todas las salidas
        group['salidas_totales_cum'] = (group['di'] + group['ci'] ).cumsum()
        
        # ai: Activos AL FINAL de la semana (los que siguen inscritos)
        group['ai'] = group['n_total'] - group['salidas_totales_cum']
        
        # ni: En riesgo AL INICIO de la semana
        # Es igual a los activos del final de la semana pasada + las salidas de esta semana
        # (porque para poder salir esta semana, tenías que estar en riesgo al inicio)
        group['ni'] = group['ai'] + group['ci'] + group['di']
        
        return group

    df_agrupado = df_agrupado.groupby(group_cols , group_keys=True).apply(calcular_flujos).reset_index()

    # Asegurar que ni no sea menor que di por errores de redondeo o data
    df_agrupado['ni'] = df_agrupado['ni'].clip(lower=0)
    grouped = df_agrupado.groupby(group_cols)

    # 4. Kaplan-Meier
    # qi: probabilidad de "morir" (desertar) dado que estás en riesgo
    df_agrupado['qi'] = np.where(df_agrupado['ni'] > 0, df_agrupado['di'] / df_agrupado['ni'], 0)
    df_agrupado['pi'] = 1 - df_agrupado['qi']
    df_agrupado['km'] = grouped['pi'].cumprod()

    # 5. Greenwood (Varianza)
    # El denominador de Greenwood es ni * (ni - di)
    mask = (df_agrupado['ni'] > df_agrupado['di']) & (df_agrupado['ni'] > 0)
    df_agrupado['greenwood_term'] = np.where(
    mask, 
    df_agrupado['di'] / (df_agrupado['ni'] * (df_agrupado['ni'] - df_agrupado['di'])), 
    0
    )
    df_agrupado['km_se'] = np.sqrt(df_agrupado['km']**2 * grouped['greenwood_term'].cumsum())

    # 6. Intervalos de Confianza
    df_agrupado['km_ic_inf'] = (df_agrupado['km'] - (conf_z * df_agrupado['km_se'])).clip(0, 1)
    df_agrupado['km_ic_sup'] = (df_agrupado['km'] + (conf_z * df_agrupado['km_se'])).clip(0, 1)

    # 7. ETI (Eficiencia Terminal)
    # Denominador: n_total - graduados/egresados acumulados previos
    # Para mayor precisión usamos el grouped real
    gi_engi_cum = grouped.apply(lambda x: (x['ci']).cumsum()).reset_index(level=0, drop=True)

    gi_engi_prev = gi_engi_cum.groupby(df_agrupado[group_cols[0]]).shift(0).fillna(0)
    denominador_eti = df_agrupado['n_total'] - gi_engi_prev

    df_agrupado['eti'] = np.where(denominador_eti > 0, df_agrupado['ai'] / denominador_eti, 0)

    # Selección final
    cols_finales = group_cols + [
    'semana_acumulada', 'n_total','ni', 'di','salidas_totales_cum', 'gi', 'engi', 'ci', 'ai', 'qi','pi',
    'km', 'km_se', 'km_ic_inf', 'km_ic_sup', 'eti'
    ]
    
    return df_agrupado[cols_finales],gi_engi_prev

In [193]:
columnas = ['cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel',
       'programa',  'fecha_inicio', 'fecha_fin', 'semana',
       'semana_acumulada','semana_limite',  'month', 'mes_academico', 'nuevos', 'di',
       'gi', 'engi','ci' ]
group_cols = ['cohorte_inicial', 'fecha_ingreso', 'nivel_academico', 'nivel',
                    'programa', ]

conf_z = 1.96

In [194]:
df = cascadas_semanal_podada_censuras.loc[:,columnas]

In [196]:
conf_z = 1.96

# Programa

In [213]:
central_tabla_vida, censuras = calcular_km_y_eti_dinamico(
    df = cascadas_semanal_podada_censuras.loc[:,columnas] , 
    group_cols = group_cols
)

In [215]:
mask_cohorte = central_tabla_vida['cohorte_inicial'] == '2025 1a'
mask_nivel = central_tabla_vida['nivel_academico'] == 'posgrado'
mask_programa = central_tabla_vida['programa'] == 'auditoria y control'
central_tabla_vida[mask_cohorte & mask_nivel & mask_programa]

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,semana_acumulada,n_total,ni,di,salidas_totales_cum,...,engi,ci,ai,qi,pi,km,km_se,km_ic_inf,km_ic_sup,eti
0,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,0,29,29.0,0.0,0.0,...,0.0,0.0,29.0,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000
1,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,1,29,29.0,0.0,0.0,...,0.0,0.0,29.0,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000
2,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,2,29,29.0,0.0,0.0,...,0.0,0.0,29.0,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000
3,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,3,29,29.0,2.0,2.0,...,0.0,0.0,27.0,0.068966,0.931034,0.931034,0.047054,0.838808,1.000000,0.931034
4,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,4,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034
5,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,5,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034
6,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,6,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034
7,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,7,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034
8,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,8,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034
9,2025 1a,2025-03-17,posgrado,especializacion,auditoria y control,9,29,27.0,0.0,2.0,...,0.0,0.0,27.0,0.000000,1.000000,0.931034,0.047054,0.838808,1.000000,0.931034


# Nivel Académico

In [216]:
central_tabla_vida, censuras = calcular_km_y_eti_dinamico(
    df = cascadas_semanal_podada_censuras.loc[:,columnas] , 
    group_cols =  ['cohorte_inicial', 'fecha_ingreso',  'nivel_academico']
) 

In [ ]:
censuras = censuras.reset_index()


,fecha_ingreso,nivel_academico,level_2,ci
0,2025-03-17,posgrado,0,0.0
1,2025-03-17,posgrado,1,0.0
2,2025-03-17,posgrado,2,0.0
3,2025-03-17,posgrado,3,0.0
4,2025-03-17,posgrado,4,0.0
...,...,...,...,...
326,2026-03-16,pregrado,326,0.0
327,2026-03-16,pregrado,327,0.0
328,2026-03-16,pregrado,328,0.0
329,2026-03-16,pregrado,329,0.0


In [ ]:
censuras['fecha_ingreso'] == '2025-03-17'

In [222]:
mask_cohorte = central_tabla_vida['cohorte_inicial'] == '2025 1c'
mask_nivel = central_tabla_vida['nivel_academico'] == 'pregrado'
#mask_programa = central_tabla_vida['programa'] == 'auditoria y control'
central_tabla_vida.loc[mask_cohorte & mask_nivel  ,['cohorte_inicial', 'fecha_ingreso',  'nivel_academico'] + ['semana_acumulada','n_total','ni', 'di','salidas_totales_cum', 'gi', 'engi', 'ci', 'ai', 'qi','pi',
'km', 'km_se', 'km_ic_inf', 'km_ic_sup', 'eti']] 

,cohorte_inicial,fecha_ingreso,nivel_academico,semana_acumulada,n_total,ni,di,salidas_totales_cum,gi,engi,ci,ai,qi,pi,km,km_se,km_ic_inf,km_ic_sup,eti
140,2025 1c,2025-07-07,pregrado,0,36,36.0,0.0,0.0,0.0,0.0,0.0,36.0,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000
141,2025 1c,2025-07-07,pregrado,1,36,36.0,1.0,1.0,0.0,0.0,0.0,35.0,0.027778,0.972222,0.972222,0.027389,0.918539,1.000000,0.972222
142,2025 1c,2025-07-07,pregrado,2,36,35.0,0.0,1.0,0.0,0.0,0.0,35.0,0.000000,1.000000,0.972222,0.027389,0.918539,1.000000,0.972222
143,2025 1c,2025-07-07,pregrado,3,36,35.0,0.0,1.0,0.0,0.0,0.0,35.0,0.000000,1.000000,0.972222,0.027389,0.918539,1.000000,0.972222
144,2025 1c,2025-07-07,pregrado,4,36,35.0,1.0,2.0,0.0,0.0,0.0,34.0,0.028571,0.971429,0.944444,0.038177,0.869618,1.000000,0.944444
145,2025 1c,2025-07-07,pregrado,5,36,34.0,0.0,2.0,0.0,0.0,0.0,34.0,0.000000,1.000000,0.944444,0.038177,0.869618,1.000000,0.944444
146,2025 1c,2025-07-07,pregrado,6,36,34.0,0.0,2.0,0.0,0.0,0.0,34.0,0.000000,1.000000,0.944444,0.038177,0.869618,1.000000,0.944444
147,2025 1c,2025-07-07,pregrado,7,36,34.0,0.0,2.0,0.0,0.0,0.0,34.0,0.000000,1.000000,0.944444,0.038177,0.869618,1.000000,0.944444
148,2025 1c,2025-07-07,pregrado,8,36,34.0,0.0,2.0,0.0,0.0,0.0,34.0,0.000000,1.000000,0.944444,0.038177,0.869618,1.000000,0.944444
149,2025 1c,2025-07-07,pregrado,9,36,34.0,0.0,2.0,0.0,0.0,0.0,34.0,0.000000,1.000000,0.944444,0.038177,0.869618,1.000000,0.944444
